# Clase 7 - Mié 29/4

## Funciones de Agrupación y Agrupamiento

- Uso de groupby y pivot_table en Pandas.

---

• Actividad: Agrupar datos y crear tablas dinámicas para resumir
información clave.

#### Actividad 1: Agrupando datos para Ventas

- 1. Cargar el conjunto de datos llamado ventas.csv, que contiene información sobre las ventas de diferentes productos en distintos meses.

- 2. Agrupar las ventas por producto y sumar las ventas totales para cada uno.

- 3. Generar un nuevo DataFrame que contenga el total de ventas por producto y lo mostrarlo en pantalla.

In [ ]:
import pandas as pd
#1 Cargar el conjunto de datos llamado ventas.csv, que contiene información sobre las ventas de diferentes productos en distintos meses.
df_ventas = pd.read_csv('/content/drive/MyDrive/datasets/ventas.csv')
df_ventas.head()

#2 Agrupar las ventas por producto y sumar las ventas totales para cada uno.
#CASTEO A FLOAT Y QUITO EL $
df_ventas['precio'] = df_ventas['precio'].str.replace('$', '', regex=False).str.strip().astype(float)

#CREO TOTAL DE VENTA
df_ventas['total_venta'] = df_ventas['precio'] * df_ventas['cantidad']
df_ventas.head()

#3

# Agrupamos por la columna 'producto' y sumamos la columna 'total_venta'
df_ventas_producto = df_ventas.groupby('producto')['total_venta'].sum().reset_index()


# Cambia el nombre de las columnas para que sea más claro
df_ventas_producto.columns = ['producto', 'monto_total_acumulado']

# Ahora df_ventas_producto es un nuevo DataFrame independiente
print(df_ventas_producto.head())

          producto  monto_total_acumulado
0  Adorno de pared               48703.24
1         Alfombra               45617.84
2       Aspiradora               51042.82
3      Auriculares               76468.44
4         Batidora               50979.20


#### Actividad 2: Creación de Tablas Dinámicas para Resumir Datos

- 1. Aplicar la función pivot_table() de Pandas para resumir información clave de los datos de ventas.

- 2. Crear un resumen que permita analizar las ventas mensuales por producto.

- 3. Interpretar los resultados y prepararlos para una presentación.

In [ ]:
#1 aplicar pivot_table() al df
df_ventas.head()

#pivot table crea una tabla nueva
tabla = pd.pivot_table( # se asigna nombre y se llama  a funcion
    df_ventas, #se aplica sobre este df
    values = 'total_venta', #valores a usar
    index = 'categoria', #colmuna de ref
    aggfunc= 'sum' #se suma
)

#agregado x mi:
total_facturado = tabla['total_venta'].sum()# se suma el total de todas las ventas

tabla['porcentaje_sobre_total'] = ( # se divide la categoria
    tabla['total_venta'] / total_facturado * 100
)

tabla['porcentaje_sobre_total'] = tabla['porcentaje_sobre_total'].round(2) #redondeo

df_ventas.head()
tabla.head() #todo ok

print("----" * 40)

#2 ventas mensuales por producto.

#casteo de la col de string a fecha:
df_ventas['fecha_venta'] = pd.to_datetime(
    df_ventas['fecha_venta'],
    dayfirst=True
)

#se agrupan las ventas por mes:
df_ventas['mes'] = df_ventas['fecha_venta'].dt.to_period('M')


# se aplica pivot table:
ventas_por_producto = pd.pivot_table(
    df_ventas,
    values = 'total_venta',
    index = ['producto'],
    columns= 'mes',
    aggfunc='sum',
    fill_value = 0
)

#se me ocurrio agregarle la dif con el mes anterior
dif_mes_anterior = ventas_por_producto.diff(axis=1)
#ventas_por_producto.head()


resumen = pd.concat(
    [ventas_por_producto, ventas_por_producto.diff(axis=1)],
    axis=1,
    keys=['ventas_por_producto', 'dif_mes_anterior']
)
resumen = resumen.sort_index(axis=1)
resumen.head()

----------------------------------------------------------------------------------------------------------------------------------------------------------------


dif_mes_anterior                                               \
mes                      2024-01  2024-02  2024-03  2024-04  2024-05  2024-06   
producto                                                                        
Adorno de pared              NaN    93.95 -1249.36  -366.56  2138.82   374.12   
Alfombra                     NaN   772.94  3214.00   927.14 -1379.04 -3515.42   
Aspiradora                   NaN  -153.22 -1025.29  1549.57 -1240.57   158.50   
Auriculares                  NaN -1088.57   747.79  2337.09 -3738.24 -1418.73   
Batidora                     NaN   481.02   246.44  1999.66 -2094.21  -527.17   

                                                     ... ventas_por_producto  \
mes              2024-07  2024-08  2024-09  2024-10  ...             2024-03   
producto                                             ...                       
Adorno de pared -2354.52  3113.37 -2942.54   281.04  ...             3355.46   
Alfombra         3266.19  -979.37 -1016.06  -574.54  ...             5629.61   
Aspiradora        788.64    80.11  -504.48  1396.77  ...             3307.41   
Auriculares      2830.44 -2463.14  1453.11 -3593.27  ...             7829.48   
Batidora        -1176.61  2685.42   573.05    -7.68  ...             3877.02   

                                                                        \
mes               2024-04  2024-05  2024-06  2024-07  2024-08  2024-09   
producto                                                                 
Adorno de pared   2988.90  5127.72  5501.84  3147.32  6260.69  3318.15   
Alfombra          6556.75  5177.71  1662.29  4928.48  3949.11  2933.05   
Aspiradora        4856.98  3616.41  3774.91  4563.55  4643.66  4139.18   
Auriculares      10166.57  6428.33  5009.60  7840.04  5376.90  6830.01   
Batidora          5876.68  3782.47  3255.30  2078.69  4764.11  5337.16   

                                            
mes              2024-10  2024-11  2024-12  
producto                                    
Adorno de pared  3599.19  2805.56  3482.72  
Alfombra         2358.51  4269.04  4095.01  
Aspiradora       5535.95  2770.80  5015.35  
Auriculares      3236.74  4555.89  3942.93  
Batidora         5329.48  4143.67  5754.48  

[5 rows x 24 columns]